# NES-VMC K4 H2/6-31G 训练（监控版）

基于对事故日志 `26-09-15-17-57_nes_vmc_K4_H2_6-31G_gauge_reset.log` 的实证诊断（用 pickle 中保存的逐步 samples + params 完整回放），结论如下：

## 事故结论：Step 219→220 的 raw 梯度爆跳（≈2 → 558 → ~1000 持续 10 步）

1. **Gauge reset 无辜（实证）**：把同一步的 `(params, samples)` 在三种规范下重算——训练 g、reset 用的 colmean、零规范——raw 梯度 / loss / tr 分布**逐位一致**（理论上有意：loss 对列规范 D_g 做相似变换，trace 不变；`dlogΨ/dθ = Tr(dL_raw/dθ)` 与 g 无关）。reset@220 只是时间上与采样事故对齐了。
2. **真正的机制是 MCMC 批次组成翻转**：活性空间极小（3200 个样本只有 ~30–150 个独立构型），训练中期 |Ψ|² 高度峰化后，16 条链的批次组成方差巨大。Step 219 起个别链状态翻转 → 逐 walker 局部能量迹 trₙ 的分布整体漂移（median −3.22 → −1.43，loss −3.21 → −2.21）→ 居中的 trₙ 与 ∇logΨ 系统性相关 → raw 梯度 ~1000；同时子空间估计退化：cond(Ŝ) ×10（2.5e4 → 3e5）、E3 变正（+0.87）、能级乱序。
3. **时间线**：219 步 E3 先偏离（先行指标，raw 仍正常）→ 220 步全面爆跳 → 230–250 步 MCMC 重新平衡后自愈。
4. **原代码的两个缺口**：报警阈值 `raw>5000` 太松（本次峰值仅 1014，从未触发）；`grad_fn` 返回的 `aux['should_skip']` 异常跳过框架存在但训练循环从未使用。

## 本 notebook 的改动
- 训练路径与原版**完全一致**（grad_fn / qgt_fn / optimizer / gauge reset 原样保留），只外挂一层监控；
- 每个 step 计算三层指标（采样健康度 / 梯度估计质量 / 物理量一致性），滚动 z 分数异常检测；
- 检测到异常步可选择**跳过参数更新**（`ANOMALY_POLICY='skip'`），让 MCMC 自愈而不吃进污染梯度；
- 附诊断面板、规范不变性自检、最差步深挖工具。

## 监控指标体系

| 层 | 指标 | 含义 / 事故中的表现 | 报警阈值 |
|---|---|---|---|
| 估计 | `ratio = raw/natural` | 自然梯度会把 1000 压到 0.1，比值放大暴露估计污染。正常 <200，事故时 >900 | > 300 |
| 估计 | `trMed`（E_L 迹中位数）滚动 z | 批次组成翻转的直接信号：−3.22 → −1.43 | z > 6 |
| 估计 | `JKρ`（jackknife 集中度） | 去掉贡献最大的 8 个 walker 后梯度范数剩余比例。本次事故中 JKρ≈1.0（污染是 tr 分布整体漂移而非个别离群），故仅作观察项，防另一类「少数 walker 主导」故障 | < 0.10 |
| 物理 | `cond(Ŝ)` | 广义本征问题重叠矩阵条件数，事故时 ×10 | > 1e5 |
| 物理 | 能级跳变滚动 z | E3: −0.55 → +0.87；能级乱序/突变的兜底信号 | z > 8 |
| 采样 | `n_uniq` / `maxDup` / 链内独立构型 | 冻结链与重复构型（本事故中区分度弱，作慢变量观察） | maxDup > 0.25 |
| 采样 | `logΨ max−min` 极差 | \|Ψ\|² 峰化程度（慢变量，峰化越强链越易被锁） | 观察 |
| 数值 | `cond(Ψ)` 全 walker max、`L_stable` min、valid 比例 | 溢出/奇病态通道（本次实测健康：cond≤500） | should_skip |
| 规范 | G 漂移速率、\|g\|、reset 吸收量 | 排除规范通道（本次自检证明 reset 数值无害） | 观察 |

In [ ]:
import logging
import os
import time
import pickle
from collections import deque

import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import optax
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from NES_VMC_V1 import (
    NESTotalAnsatz_stable,
    create_single_machine_gauge_fixed,
    Ham_Psi_scaled,
    flatten_batched_pytree,
    NESFermionHopRule,
    ravel_pytree,
)
from NES_VMC_tool import (
    create_gauge_reset_total_machines,
    NES_loss_energy_stable_gauge,
    make_grad_fn_gauge,
    make_qgt_fn_gauge,
    make_gauge_fn,
    make_MS_estimator_fn,
    compute_lam_v_from_samples,
)
from H2_631G import SINGLE_SIZE, ha, hi_ext, ext_edges, K, Hatree_Fock, hi, E_fcis

# ====================== 日志配置 ======================
time_str = time.strftime('%y-%m-%d-%H-%M')
logger = logging.getLogger('NES_VMC_K4_gauge_reset_monitored')
logger.setLevel(logging.INFO)
logger.propagate = False
logger.handlers.clear()
simple_formatter = logging.Formatter('%(asctime)s %(message)s', datefmt='%y-%d-%H-%M')
os.makedirs('./日志', exist_ok=True)
log_path = f'./日志/{time_str}_nes_vmc_K4_H2_6-31G_gauge_reset_monitored.log'
file_handler = logging.FileHandler(log_path, mode='w', encoding='utf-8')
file_handler.setFormatter(simple_formatter)
logger.addHandler(file_handler)
console_handler = logging.StreamHandler()
console_handler.setFormatter(simple_formatter)
logger.addHandler(console_handler)

# ====================== 超参配置（与事故运行一致） ======================
N_CHAINS = 16
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 10
N_ITER = 300
Natural_Grad = True
clip_norm = 20.0
lr = 0.1
qgt_diag_shift = 0.1
RESET_PERIOD = 10
SAVE_INTERVAL = 20
HISTORY_FILE = f'./data/{time_str}_history_monitored_H2_molecule_K4.pkl'
os.makedirs('./data', exist_ok=True)

# ====================== 监控/处置配置（本次新增） ======================
ANOMALY_POLICY = 'skip'   # 'skip': 异常步跳过参数更新 | 'none': 仅记录不干预
WARMUP_STEPS = 20         # 预热期只记录不报警
RATIO_MAX = 300.0         # raw/natural 比值阈值（正常 <200，事故 >900）
TR_Z_MAX = 6.0            # E_L 迹中位数滚动 z 阈值
LEVEL_Z_MAX = 8.0         # 能级跳变滚动 z 阈值
CONDS_MAX = 1e5           # cond(Ŝ) 阈值
JK_MIN = 0.10             # 梯度 jackknife 集中度下限（观察项）
MAX_DUP_FRAC = 0.25       # 单构型最大重复占比
ROLL_WIN = 50             # 滚动统计窗口
JK_TOP_K = 8              # jackknife 剔除的贡献最大 walker 数

logger.info('=' * 60)
logger.info('开始多链 NES-VMC 训练 | 监控版: 三层指标 + 滚动 z 异常检测 + skip 处置')
logger.info(
    f'超参: N_CHAINS={N_CHAINS}, N_SAMPLES_PER_CHAIN={N_SAMPLES_PER_CHAIN}, SWEEP_SIZE={SWEEP_SIZE}, '
    f'N_ITER={N_ITER}, clip_norm={clip_norm}, lr={lr}, qgt_diag_shift={qgt_diag_shift}, RESET_PERIOD={RESET_PERIOD}'
)
logger.info(
    f'监控: policy={ANOMALY_POLICY}, RATIO_MAX={RATIO_MAX}, TR_Z_MAX={TR_Z_MAX}, '
    f'LEVEL_Z_MAX={LEVEL_Z_MAX}, CONDS_MAX={CONDS_MAX}, JK_MIN={JK_MIN}, MAX_DUP_FRAC={MAX_DUP_FRAC}'
)
logger.info('=' * 60)

In [ ]:
# ====================== 模型与采样器（与事故运行一致） ======================
total_ansatz = NESTotalAnsatz_stable(
    n_spin_orbitals=SINGLE_SIZE,
    n_states=K,
    hidden_dim=SINGLE_SIZE + K,
    rngs=nnx.Rngs(11),
)

g_current = jnp.zeros(K, dtype=jnp.complex64)

(
    total_machine,
    total_matrix_machine,
    total_max_machine,
    total_matrix_machine_raw,
    total_graphdef,
    total_params,
) = create_gauge_reset_total_machines(total_ansatz, Hatree_Fock)

single_machine_list = [
    create_single_machine_gauge_fixed(ansatz, Hatree_Fock)[0]
    for ansatz in total_ansatz.single_ansatz_list
]

grad_fn = make_grad_fn_gauge(
    ha,
    total_matrix_machine,
    total_max_machine,
    total_machine,
    single_machine_list,
)
qgt_fn = make_qgt_fn_gauge(total_machine)
gauge_fn, col_mean_fn = make_gauge_fn(total_ansatz, Hatree_Fock)
energy_estimator = make_MS_estimator_fn(ha=ha, single_machine_list=single_machine_list)

nes_rule = NESFermionHopRule(edges=ext_edges, K=K, single_size=SINGLE_SIZE)
nes_sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_ext,
    rule=nes_rule,
    n_chains=N_CHAINS,
    sweep_size=SWEEP_SIZE,
)

exact_eigvals = E_fcis
logger.info(
    f'精确CAS基准: ' + ' | '.join(f'E{i}={exact_eigvals[i]:.8f}' for i in range(K))
    + f' | 理论 Loss 上限={sum(exact_eigvals[:K]):.8f}'
)

# ====================== 优化器 ======================
optimizer = optax.chain(
    optax.clip_by_global_norm(clip_norm),
    optax.sgd(learning_rate=lr),
)
opt_state = optimizer.init(total_params)

sampler_rng = jax.random.PRNGKey(21)

def sample_machine(params, sigma):
    return total_machine(params, sigma, g_current)

sampler_state = nes_sampler.init_state(sample_machine, total_params, sampler_rng)

In [ ]:
# ====================== 监控工具 ======================
grad_logpsi_mon = jax.grad(total_machine, argnums=0, holomorphic=True)
vmap_grad_logpsi_mon = jax.vmap(grad_logpsi_mon, in_axes=(None, 0, None))

@jax.jit
def monitor_fn(params, x_batch, g):
    '''一次前向+反向拿到全部监控原始量:
    tr_n   (N,)    逐 walker 的 E_L 迹（事故中批次翻转时整体漂移的直接信号）
    O      (N, P)  逐 walker 的 dlogΨ 展平矩阵（QGT 同款，用于集中度分析）
    condΨ  (N,)    全 walker 的 Ψ 条件数（原版只看 walker0，会漏检）
    shift  (N,)    log-det 稳定化平移
    '''
    loss_batch, E_L_batch, laux = NES_loss_energy_stable_gauge(
        ha=ha,
        total_matrix_machine=total_matrix_machine,
        total_max_machine=total_max_machine,
        single_machine_list=single_machine_list,
        total_params=params,
        x=x_batch,
        g=g,
        return_aux=True,
    )
    E_L = jnp.linalg.solve(laux['Psi_Matrix_stable'], laux['HPsi_stable'])
    tr_n = jnp.real(jnp.trace(E_L, axis1=-2, axis2=-1))
    O = flatten_batched_pytree(vmap_grad_logpsi_mon(params, x_batch, g), x_batch.shape[0])
    return tr_n, O, laux['cond_Psi'], laux['shift']

class RollingZ:
    '''滚动中位数/MAD 的 z 分数；窗口未满（预热期）返回 0。'''
    def __init__(self, win=ROLL_WIN):
        self.buf = deque(maxlen=win)

    def score(self, x):
        if len(self.buf) < WARMUP_STEPS:
            return 0.0
        arr = np.asarray(self.buf)
        med = np.median(arr)
        mad = 1.4826 * np.median(np.abs(arr - med)) + 1e-12
        return abs(float(x) - med) / mad

    def update(self, x):
        z = self.score(x)
        self.buf.append(float(x))
        return z

def sampling_health(x_batch):
    '''采样健康度: 独立构型数 / 单构型最大重复占比 / 逐链独立构型数。'''
    flat = x_batch.reshape(len(x_batch), -1)
    uniq, counts = np.unique(flat, axis=0, return_counts=True)
    per_chain = [
        len(np.unique(flat[c * N_SAMPLES_PER_CHAIN:(c + 1) * N_SAMPLES_PER_CHAIN], axis=0))
        for c in range(N_CHAINS)
    ]
    return {
        'n_uniq': int(len(uniq)),
        'max_dup_frac': float(counts.max() / len(flat)),
        'min_chain_uniq': int(min(per_chain)),
        'mean_chain_uniq': float(np.mean(per_chain)),
    }

def gradient_concentration(tr_n, O, top_k=JK_TOP_K):
    '''jackknife 集中度: 去掉 |tr_c·conj(O)| 行范数最大的 top_k 个 walker 后，
    梯度范数剩余比例 JKρ 与方向夹角 cos。JKρ << 1 说明梯度被少数 walker 主导
    （事故 step220-229 的直接成因）。'''
    tr_c = tr_n - tr_n.mean()
    v = tr_c[:, None] * np.conj(O)
    g_full = v.mean(axis=0)
    row_norm = np.linalg.norm(v, axis=1)
    idx = np.argsort(row_norm)[::-1][:top_k]
    g_excl = np.delete(v, idx, axis=0).mean(axis=0)
    n_full = np.linalg.norm(g_full) + 1e-30
    n_excl = np.linalg.norm(g_excl)
    ratio = float(n_excl / n_full)
    cos = float(np.real(np.vdot(g_excl, g_full)) / (n_excl * n_full + 1e-30))
    return ratio, cos, idx

In [ ]:
# ====================== 历史容器 ======================
loss_history, logpsi_history, steps_history = [], [], []
logpsi_mean_history, logpsi_min_history, logpsi_max_history = [], [], []
grad_norm_raw_history, grad_norm_nat_history = [], []
E_L_real_history, E_L_imag_history = [], []
params_history, Energy_levels_history, samples_history = [], [], []
ratio_history = []
tr_med_history, tr_p25_history, tr_p75_history = [], [], []
jk_ratio_history, jk_cos_history = [], []
condS_history, condpsi_max_history = [], []
uniq_history, dup_history, chain_uniq_history = [], [], []
gauge_drift_history, g_abs_history = [], []
flag_history, action_history = [], []
skipped_steps = []

z_tr = RollingZ()
z_level = [RollingZ() for _ in range(K)]

def build_history():
    return {
        'steps': [int(s) for s in steps_history],
        'logpsi_mean': logpsi_mean_history,
        'logpsi_min': logpsi_min_history,
        'logpsi_max': logpsi_max_history,
        'grad_norm_raw': grad_norm_raw_history,
        'grad_norm_natural': grad_norm_nat_history,
        'ratio': ratio_history,
        'tr_med': tr_med_history,
        'tr_p25': tr_p25_history,
        'tr_p75': tr_p75_history,
        'jk_ratio': jk_ratio_history,
        'jk_cos': jk_cos_history,
        'cond_S': condS_history,
        'cond_psi_max': condpsi_max_history,
        'n_uniq': uniq_history,
        'max_dup_frac': dup_history,
        'mean_chain_uniq': chain_uniq_history,
        'gauge_drift': gauge_drift_history,
        'g_abs': g_abs_history,
        'flags': flag_history,
        'action': action_history,
        'skipped_steps': skipped_steps,
        'E_L_real': E_L_real_history,
        'E_L_imag': E_L_imag_history,
        'loss': loss_history,
        'first_step': 0,
        'save_interval': SAVE_INTERVAL,
        'Energy_levels': Energy_levels_history,
        'params': params_history,
        'samples': samples_history,
    }

# ====================== 训练主循环 ======================
start_time = time.time()

for step in range(N_ITER):
    params_history.append(total_params)

    # 1. 采样
    samples_raw, sampler_state = nes_sampler.sample(
        machine=sample_machine,
        parameters=total_params,
        state=sampler_state,
        chain_length=N_SAMPLES_PER_CHAIN,
    )
    samples = samples_raw.reshape(-1, hi_ext.size)
    x_batch = samples.reshape(-1, K, SINGLE_SIZE)

    # 2. 梯度与损失（与原版完全一致）
    grad_raw, loss_mean, E_L_mean, aux = grad_fn(total_params, x_batch, g_current)
    grad_raw_flat, unravel_fn = ravel_pytree(grad_raw)
    grad_norm_raw = float(jnp.linalg.norm(grad_raw_flat))

    # 3. 监控数据（外挂，不影响优化路径）
    tr_n, O, cond_psi_all, shift_b = monitor_fn(total_params, x_batch, g_current)
    tr_n_np = np.asarray(tr_n)
    tr_med = float(np.median(tr_n_np))
    tr_p25 = float(np.percentile(tr_n_np, 25))
    tr_p75 = float(np.percentile(tr_n_np, 75))
    jk_ratio, jk_cos, top_idx = gradient_concentration(tr_n_np, np.asarray(O))
    sh = sampling_health(np.asarray(samples).reshape(-1, K, SINGLE_SIZE))
    Mhat, Shat, finite_ms = energy_estimator(total_params, x_batch)
    cond_S = float(np.linalg.cond(np.asarray(Shat)))
    lam_real, v_eig, n_valid_eig, order = compute_lam_v_from_samples(
        ha=ha,
        single_machine_list=single_machine_list,
        total_params=total_params,
        x_batch=x_batch,
        estimator=energy_estimator,
    )
    eig_vals = lam_real
    z_tr_val = z_tr.update(tr_med)
    z_lvl = [z.update(float(eig_vals[i])) for i, z in enumerate(z_level)]

    # 4. QGT 自然梯度（与原版完全一致）
    if Natural_Grad:
        qgt_reg_mat = qgt_fn(total_params, x_batch, qgt_diag_shift, g_current)
        ng_flat = jnp.linalg.solve(qgt_reg_mat, grad_raw_flat)
        grad_norm_natural = float(jnp.linalg.norm(ng_flat))
    else:
        ng_flat = grad_raw_flat
        grad_norm_natural = grad_norm_raw

    ratio = grad_norm_raw / max(grad_norm_natural, 1e-12)

    # 5. 异常检测（滚动 z + 硬阈值 + nan/valid 兜底）
    flags = []
    if bool(aux['should_skip']):
        flags.append('invalidBatch')
    if step >= WARMUP_STEPS:
        if ratio > RATIO_MAX:
            flags.append('ratio')
        if z_tr_val > TR_Z_MAX:
            flags.append('trShift')
        if any(z > LEVEL_Z_MAX for z in z_lvl):
            flags.append('levelJump')
        if cond_S > CONDS_MAX:
            flags.append('condS')
        if jk_ratio < JK_MIN:
            flags.append('gradConc')
        if sh['max_dup_frac'] > MAX_DUP_FRAC:
            flags.append('dupFrac')
    anomaly = len(flags) > 0

    # 6. 更新 / 跳过（异常步不吃进污染梯度，让 MCMC 自愈）
    action = 'UPDATE'
    if anomaly and ANOMALY_POLICY == 'skip':
        action = 'SKIP'
        skipped_steps.append(step)
    else:
        grad_update = unravel_fn(ng_flat)
        updates, opt_state = optimizer.update(grad_update, opt_state, total_params)
        total_params = optax.apply_updates(total_params, updates)
    grad_norm_clipped = min(grad_norm_natural, clip_norm)

    # 7. 波函数与规范监控（原版口径）
    log_Psi_batch = total_machine(total_params, x_batch, g_current)
    x_single = x_batch[0:1, ...]
    psi_mat = total_matrix_machine(total_params, x_single, g_current)[0]
    psi_cond = jnp.linalg.cond(psi_mat)
    gauge_now = float(jnp.real(gauge_fn(total_params, x_batch)))
    g_abs = float(jnp.linalg.norm(g_current))
    cond_psi_max = float(np.max(np.asarray(cond_psi_all)))

    # 8. 日志（前 5 行与原版格式对齐，后 3 行为监控扩展）
    logger.info(f'[Step {step:3d}] logΨ: mean={log_Psi_batch.mean():.3f} | min={log_Psi_batch.min():.3f} | max={log_Psi_batch.max():.3f}')
    logger.info(f'梯度监控 | raw={grad_norm_raw:.4f} | natural={grad_norm_natural:.4f} | clipped={grad_norm_clipped:.4f}(上限{clip_norm})')
    logger.info(f'原始规范坐标 G(Re Σrow_mean) = {gauge_now:+.4f} | 累计|g| = {g_abs:.4f}')
    logger.info(f'Ψ矩阵条件数 cond(Ψ) = {psi_cond:.2e} | 全walker max={cond_psi_max:.2e}')
    logger.info(
        f'Loss={float(jnp.real(loss_mean)):.6f} | E0={eig_vals[0]:.8f} | E1={eig_vals[1]:.8f}'
        f'|E2={eig_vals[2]:.8f}|E3={eig_vals[3]:.8f}'
    )
    logger.info(
        f"采样监控 | uniq={sh['n_uniq']} maxDup={sh['max_dup_frac']:.2%} "
        f"minChainUniq={sh['min_chain_uniq']} meanChainUniq={sh['mean_chain_uniq']:.1f}"
    )
    logger.info(
        f'估计监控 | trMed={tr_med:.4f} trIQR={tr_p75 - tr_p25:.4f} zTr={z_tr_val:.1f} '
        f'JKρ={jk_ratio:.3f} cos={jk_cos:.3f} condS={cond_S:.2e} ratio={ratio:.1f}'
    )
    logger.info(f"状态 | flags={'.'.join(flags) if flags else 'ok'} | action={action}")
    logger.info('#-----------------------------------------#')

    # 9. 历史记录
    steps_history.append(step)
    loss_history.append(float(jnp.real(loss_mean)))
    logpsi_history.append(float(jnp.real(log_Psi_batch.mean())))
    logpsi_mean_history.append(float(jnp.real(log_Psi_batch.mean())))
    logpsi_min_history.append(float(jnp.real(log_Psi_batch.min())))
    logpsi_max_history.append(float(jnp.real(log_Psi_batch.max())))
    grad_norm_raw_history.append(grad_norm_raw)
    grad_norm_nat_history.append(grad_norm_natural)
    ratio_history.append(ratio)
    E_L_real_history.append(float(jnp.real(jnp.trace(E_L_mean))))
    E_L_imag_history.append(float(jnp.imag(jnp.trace(E_L_mean))))
    Energy_levels_history.append(eig_vals[:K])
    samples_history.append(samples)
    tr_med_history.append(tr_med)
    tr_p25_history.append(tr_p25)
    tr_p75_history.append(tr_p75)
    jk_ratio_history.append(jk_ratio)
    jk_cos_history.append(jk_cos)
    condS_history.append(cond_S)
    condpsi_max_history.append(cond_psi_max)
    uniq_history.append(sh['n_uniq'])
    dup_history.append(sh['max_dup_frac'])
    chain_uniq_history.append(sh['mean_chain_uniq'])
    gauge_drift_history.append(gauge_now)
    g_abs_history.append(g_abs)
    flag_history.append(flags)
    action_history.append(action)

    # 10. 周期保存
    if (step + 1) % SAVE_INTERVAL == 0:
        with open(HISTORY_FILE, 'wb') as f:
            pickle.dump(build_history(), f)
        logger.info(f'[保存] Step {step} -> pickle 文件已更新 ({len(steps_history)} 条记录)')

    # 11. 周期性 gauge reset（与原版一致；自检已证明其数值无害）
    if (step + 1) % RESET_PERIOD == 0:
        new_col_mean = col_mean_fn(total_params, x_batch)
        g_current = new_col_mean
        logger.info(
            f'[GaugeReset@{step + 1}] 吸收 Δg = Σ Re(col_mean) = {float(jnp.sum(jnp.real(new_col_mean))):+.4f} '
            f'| 新|g| = {float(jnp.linalg.norm(g_current)):.4f}'
        )

end_time = time.time()
logger.info(f'训练耗时: {end_time - start_time:.2f} 秒')
logger.info(f'跳过的异常步: {skipped_steps}')
with open(HISTORY_FILE, 'wb') as f:
    pickle.dump(build_history(), f)
logger.info(f'[保存] 训练结束, pickle 最终写入 {len(steps_history)} 条记录 -> {HISTORY_FILE}')

In [ ]:
# ====================== 诊断面板 ======================
plt.rcParams.update({
    'font.family': 'Arial',
    'mathtext.fontset': 'cm',
    'axes.labelsize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
})

st = np.array(steps_history)
E_arr = np.array(Energy_levels_history)
reset_lines = list(range(RESET_PERIOD, N_ITER + 1, RESET_PERIOD))

def _deco(ax, title, ylog=False):
    for r in reset_lines:
        ax.axvline(r - 0.5, color='gray', ls=':', lw=0.6, alpha=0.7)
    for s in skipped_steps:
        ax.axvline(s, color='red', lw=1.0, alpha=0.5)
    if ylog:
        ax.set_yscale('log')
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.25)

fig, axes = plt.subplots(3, 3, figsize=(18, 12))

ax = axes[0, 0]
ax.plot(st, loss_history, lw=1)
ax.axhline(sum(exact_eigvals[:K]), color='k', ls='--', lw=0.8, label='exact ΣE')
ax.legend()
_deco(ax, 'Loss (红竖线=skip 步, 灰点线=reset)')

ax = axes[0, 1]
ax.plot(st, grad_norm_raw_history, label='raw', lw=1)
ax.plot(st, grad_norm_nat_history, label='natural', lw=1)
ax.legend()
_deco(ax, '梯度范数', ylog=True)

ax = axes[0, 2]
ax.plot(st, ratio_history, lw=1)
ax.axhline(RATIO_MAX, color='red', ls='--', lw=0.8)
_deco(ax, 'raw/natural 比值 (事故主报警)', ylog=True)

ax = axes[1, 0]
ax.plot(st, tr_med_history, lw=1, label='tr median')
ax.fill_between(st, tr_p25_history, tr_p75_history, alpha=0.3, label='IQR')
ax.legend()
_deco(ax, 'E_L 迹分布 (批次组成翻转的直接信号)')

ax = axes[1, 1]
for i in range(K):
    ax.plot(st, E_arr[:, i], lw=1, label=f'$E_{i}$')
    ax.axhline(exact_eigvals[i], ls='--', lw=0.8, color=f'C{i}')
ax.set_ylim(-1.5, 0.3)
ax.legend(loc='lower right')
_deco(ax, '能级 vs CAS 精确值 (虚线)')

ax = axes[1, 2]
ax.plot(st, condS_history, lw=1)
ax.axhline(CONDS_MAX, color='red', ls='--', lw=0.8)
_deco(ax, 'cond(Ŝ) 子空间重叠矩阵', ylog=True)

ax = axes[2, 0]
ax.plot(st, uniq_history, lw=1, label='独立构型数')
ax.plot(st, np.array(dup_history) * 100, lw=1, label='最大重复占比 %')
ax.legend()
_deco(ax, '采样多样性')

ax = axes[2, 1]
ax.plot(st, jk_ratio_history, lw=1, label='JKρ 集中度')
ax.plot(st, jk_cos_history, lw=1, label='cos 方向一致性')
ax.axhline(JK_MIN, color='red', ls='--', lw=0.8)
ax.legend()
_deco(ax, '梯度贡献集中度 jackknife')

ax = axes[2, 2]
ax.plot(st, gauge_drift_history, lw=1, label='G (raw 行规范坐标)')
ax.plot(st, g_abs_history, lw=1, label='|g|')
ax.legend()
_deco(ax, '规范坐标 (reset 吸收/漂移)')

fig.tight_layout()
panel_path = f'./日志/[监控面板]{time_str}.png'
fig.savefig(panel_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'诊断面板已保存: {panel_path}')
print(f'异常步 (flags 非空): {[s for s, f in zip(steps_history, flag_history) if f]}')
print(f'skip 处置步: {skipped_steps}')

## 附录 A：规范不变性自检

事故诊断中的关键实验：同一 `(params, samples)` 在三种规范（训练 g / reset 用的 colmean / 零规范）下重算 raw 梯度与 loss。
理论上 loss 对列规范做相似变换 trace 不变、`dlogΨ/dθ = Tr(dL_raw/dθ)` 与 g 无关；数值上也应逐位一致。若出现明显差异，说明出现了数值病态（如某列 exp 下溢），需要立即处理。

In [ ]:
# 用最后一步的 params 与 samples 做三规范对照
p_last = params_history[-1]
x_last = jnp.asarray(np.asarray(samples_history[-1]).reshape(-1, K, SINGLE_SIZE))
g_cm = col_mean_fn(p_last, x_last)
print(f'colmean g = {np.asarray(g_cm)}')
print(f"{'规范':>18} | {'raw 梯度范数':>12} | {'loss':>10}")
for tag, g_try in [
    ('训练 g (g_current)', g_current),
    ('colmean (reset 后)', g_cm),
    ('零规范', jnp.zeros(K, dtype=jnp.complex64)),
]:
    gr, lm, _, _ = grad_fn(p_last, x_last, g_try)
    grf, _ = ravel_pytree(gr)
    print(f'{tag:>18} | {float(jnp.linalg.norm(grf)):12.4f} | {float(jnp.real(lm)):10.6f}')

## 附录 B：最差步深挖

对 raw/natural 比值最大的步，逐 walker 拆解梯度贡献：
- 每个 walker 的贡献项为 `v_n = (tr_n − ⟨tr⟩) · conj(∇logΨ_n)`，raw 梯度 = mean(v_n)；
- 打印贡献最大的 10 个 walker：属于哪条链、tr_n 值、|v_n|、其构型在批内的重复次数（识别被锁死的链）；
- 打印各链 tr_n 均值：少数链均值明显偏离 = 批次组成翻转的直接证据。

In [ ]:
worst = int(np.argmax(ratio_history))
print(f'ratio 最大的步: {worst} (ratio={ratio_history[worst]:.1f}, flags={flag_history[worst]})')

x_w = jnp.asarray(np.asarray(samples_history[worst]).reshape(-1, K, SINGLE_SIZE))
tr_n_w, O_w, _, _ = monitor_fn(params_history[worst], x_w, g_current)
tr_n_w = np.asarray(tr_n_w)
O_w = np.asarray(O_w)

tr_c = tr_n_w - tr_n_w.mean()
v = tr_c[:, None] * np.conj(O_w)
row_norm = np.linalg.norm(v, axis=1)
top = np.argsort(row_norm)[::-1][:10]

flat = np.asarray(samples_history[worst]).reshape(len(tr_n_w), -1)
uniq_arr, counts_arr = np.unique(flat, axis=0, return_counts=True)
cnt_of = {tuple(r): int(c) for r, c in zip(uniq_arr, counts_arr)}

print(f"{'walker':>7} {'链':>3} {'tr_n':>9} {'|v_n|':>10} {'构型重复次数':>8}")
for widx in top:
    dup = cnt_of.get(tuple(flat[widx]), 1)
    print(f'{widx:>7} {widx // N_SAMPLES_PER_CHAIN:>3} {tr_n_w[widx]:>9.3f} {row_norm[widx]:>10.2f} {dup:>8}')

per_chain_tr = tr_n_w.reshape(N_CHAINS, N_SAMPLES_PER_CHAIN).mean(axis=1)
print('各链 tr_n 均值:')
print(np.array2string(per_chain_tr, precision=3, max_line_width=100))
print(f'全批 tr_n 中位数 = {np.median(tr_n_w):.4f}')

## 后续可做的改进

1. **稳健梯度**：异常步不 skip 而是对 tr_c 做 winsorize（clip 到 median ± 3·MAD）后重算梯度，保留更新但压制离群 walker；
2. **采样侧**：异常连续出现时临时增大 `SWEEP_SIZE` 或对样本做去重重加权；对小活性空间系统，链数 `N_CHAINS` 比每链样本数更值得加大（本事故本质是 16 条链的组成方差）；
3. **ESS 监控**：基于构型重复度估计有效样本量 `ESS ≈ N / Σ p_i²`，作为自适应阈值；
4. **规范通道**：本实证已证明 reset 数值无害，`RESET_PERIOD` 可按 G 漂移速率自适应（漂移到阈值才 reset），减少不必要扰动；
5. **QGT 对角 shift**：`cond(Ŝ)` 长期偏高时可考虑对 QGT 特征值截断（Marzolino 替代解），提高自然梯度对污染方向的鲁棒性。